
Libraries Import
```
```



In [13]:
import pandas as pd
import numpy as np
import re
import nltk

In [14]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [15]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation


In [16]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

Dataset Load

In [17]:
df = pd.read_csv("Amazon_Reviews.csv", engine='python')

In [18]:
df.head()

,Reviewer Name,Profile Link,Country,Review Count,Review Date,Rating,Review Title,Review Text,Date of Experience
0,Eugene ath,/users/66e8185ff1598352d6b3701a,US,1 review,2024-09-16T13:44:26.000Z,Rated 1 out of 5 stars,A Store That Doesn't Want to Sell Anything,"I registered on the website, tried to order a ...","September 16, 2024"
1,Daniel ohalloran,/users/5d75e460200c1f6a6373648c,GB,9 reviews,2024-09-16T18:26:46.000Z,Rated 1 out of 5 stars,Had multiple orders one turned up and…,Had multiple orders one turned up and driver h...,"September 16, 2024"
2,p fisher,/users/546cfcf1000064000197b88f,GB,90 reviews,2024-09-16T21:47:39.000Z,Rated 1 out of 5 stars,I informed these reprobates,I informed these reprobates that I WOULD NOT B...,"September 16, 2024"
3,Greg Dunn,/users/62c35cdbacc0ea0012ccaffa,AU,5 reviews,2024-09-17T07:15:49.000Z,Rated 1 out of 5 stars,Advertise one price then increase it on website,I have bought from Amazon before and no proble...,"September 17, 2024"
4,Sheila Hannah,/users/5ddbe429478d88251550610e,GB,8 reviews,2024-09-16T18:37:17.000Z,Rated 1 out of 5 stars,If I could give a lower rate I would,If I could give a lower rate I would! I cancel...,"September 16, 2024"


In [19]:
df['Review Text'].isnull().sum()
df = df.dropna(subset=['Review Text'])

Text Preprocessing

In [20]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    words = text.split()

    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]

    return " ".join(words)

df['clean_text'] = df['Review Text'].apply(preprocess)

Vector (Bag of Words)

In [21]:
vectorizer = CountVectorizer(max_df=0.9, min_df=5)
X = vectorizer.fit_transform(df['clean_text'])

Apply LDA Model

In [ ]:
lda = LatentDirichletAllocation(n_components=5, random_state=42)

lda.fit(X)

In [ ]:
def print_topics(model, feature_names, n_words):
    for i, topic in enumerate(model.components_):
        print(f"\nTopic {i}:")
        print([feature_names[j] for j in topic.argsort()[-n_words:]])

print_topics(lda, vectorizer.get_feature_names_out(), 10)

In [ ]:
def print_topics(model, feature_names, n_words):
    for i, topic in enumerate(model.components_):
        print(f"\nTopic {i}:")
        print([feature_names[j] for j in topic.argsort()[-n_words:]])

print_topics(lda, vectorizer.get_feature_names_out(), 10)

In [ ]:
topic_values = lda.transform(X)

df['Topic'] = topic_values.argmax(axis=1)

df[['Review Text', 'Topic']].head()

In [ ]:
import matplotlib.pyplot as plt

df['Topic'].value_counts().plot(kind='bar')
plt.title("Topic Distribution")
plt.xlabel("Topic")
plt.ylabel("Count")
plt.show()